In [ ]:
!pip install pdf2image pillow opencv-python
!apt-get install poppler-utils -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (462 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
from pdf2image import convert_from_path
from google.colab import files
import os
from PIL import Image
import cv2
import numpy as np

# Upload PDF
print("Upload your voter card PDF:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Create output directory
output_dir = "voter_cards_dataset/images"
os.makedirs(output_dir, exist_ok=True)

# Convert PDF to images at high resolution
print("Converting PDF pages to images...")
images = convert_from_path(pdf_path, dpi=300)

for i, image in enumerate(images, 1):
    # Save each page as JPG
    image_path = os.path.join(output_dir, f"page_{i:03d}.jpg")
    image.save(image_path, 'JPEG', quality=95)
    print(f"  Saved: page_{i:03d}.jpg")

print(f"\n Converted {len(images)} pages to images")
print(f"Images saved to: {output_dir}")

# Download images for annotation
!zip -r voter_cards_images.zip voter_cards_dataset/images
files.download('voter_cards_images.zip')

In [ ]:
import matplotlib.pyplot as plt

def preview_pages(start=1, end=5):
    """Preview first few pages"""
    fig, axes = plt.subplots(1, end-start+1, figsize=(20, 5))
    for idx, i in enumerate(range(start, end+1)):
        img_path = f"voter_cards_dataset/images/page_{i:03d}.jpg"
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(f"Page {i}")
        axes[idx].axis('off')
    plt.tight_layout()
    plt.show()

preview_pages(1, 5)

In [ ]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
# Import libraries
from ultralytics import YOLO
import torch
import yaml
from google.colab import files
import shutil

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY") # Get API key from Roboflow..
project = rf.workspace("your_workspace_name").project("your_project_name") # Remember to UPDATE!!!
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Voter-Card-Detection-1 in yolov8:: 100%|██████████| 82/82 [00:00<00:00, 6911.70it/s]


In [ ]:
# Split into train/val (90/10)
import random
import os
from pathlib import Path

def split_dataset(images_dir, labels_dir, output_dir, train_ratio=0.9):
    """Split dataset into train and validation sets"""

    # Create directories
    for split in ['train', 'val']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)

    # Get all image files
    images = list(Path(images_dir).glob('*.jpg'))
    random.shuffle(images)

    split_idx = int(len(images) * train_ratio)
    train_images = images[:split_idx]
    val_images = images[split_idx:]

    # Copy files
    for img_path in train_images:
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy(img_path, f"{output_dir}/train/images/{img_path.name}")
            shutil.copy(label_path, f"{output_dir}/train/labels/{label_path.name}")

    for img_path in val_images:
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy(img_path, f"{output_dir}/val/images/{img_path.name}")
            shutil.copy(label_path, f"{output_dir}/val/labels/{label_path.name}")

    print(f"✓ Train set: {len(train_images)} images")
    print(f"✓ Val set: {len(val_images)} images")

# Run split
split_dataset(
    '/content/Voter-Card-Detection-1/train/images',
    '/content/Voter-Card-Detection-1/train/labels',
    'dataset_split'
)

✓ Train set: 34 images
✓ Val set: 4 images


In [ ]:
data_yaml = {
    'path': '/content/dataset_split',  # dataset root dir
    'train': 'train/images',  # train images
    'val': 'val/images',  # val images
    'nc': 1,  # number of classes
    'names': ['voter_card']  # class names
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("✓ Created data.yaml")

✓ Created data.yaml


In [ ]:
# Load pre-trained YOLOv8n model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='data.yaml',
    epochs=100,  # Increase if needed
    imgsz=640,   # Image size
    batch=16,    # Batch size (reduce if OOM)
    patience=20, # Early stopping patience
    device=0,    # GPU device (0 for first GPU)
    workers=4,   # Number of workers
    project='voter_card_detection',
    name='yolov8n_run',

    # Optimization for small objects
    mosaic=1.0,   # Mosaic augmentation
    mixup=0.1,    # Mixup augmentation
    degrees=5,    # Rotation augmentation
    translate=0.1,  # Translation augmentation
    scale=0.2,    # Scale augmentation

    # Performance
    cache=True,   # Cache images for faster training
    verbose=True,
    plots=True    # Save training plots
)

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose

In [ ]:
# Validate the model
metrics = model.val()

print(f"\nModel Performance:")
print(f"  mAP50: {metrics.box.map50:.3f}")
print(f"  mAP50-95: {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.mp:.3f}")
print(f"  Recall: {metrics.box.mr:.3f}")

# Test on sample images
!mkdir test_predictions

results = model.predict(
    source='dataset_split/val/images',
    conf=0.25,  # Confidence threshold
    iou=0.45,   # NMS IoU threshold
    save=True,
    save_txt=True,  # Save labels
    project='test_predictions'
)

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1380.8±376.4 MB/s, size: 49.3 KB)
val: Scanning /content/dataset_split/val/labels.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 1.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 8.1it/s 0.1s
                   all          4         72      0.999          1      0.995      0.994
Speed: 2.5ms preprocess, 16.3ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val

Model Performance:
  mAP50: 0.995
  mAP50-95: 0.994
  Precision: 0.999
  Recall: 1.000

image 1/4 /content/dataset_split/val/images/page_002_jpg.rf.b368f77cbf2c48a18261fa0ba215aecb.jpg: 640x640 18 voter_cards, 7.3ms
image 2/4 /content/dataset_split/val/images/page_003_jpg.rf.09bb8bd830

In [ ]:
# Export to ONNX for faster CPU inference
model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12
)

# Download trained weights
!zip -r voter_card_yolov8n.zip voter_card_detection/yolov8n_run/weights

print("\n Model training complete!")

In [ ]:
import sys
import pkg_resources

print("="*70)
print("COMPLETE ENVIRONMENT REPORT")
print("="*70)

# Python version
print(f"\n Python: {sys.version}")

# Installed packages
print("\n Python Packages:")
packages = ['pdf2image', 'pytesseract', 'Pillow', 'ultralytics', 'opencv-python-headless', 'numpy', 'torch', 'torchvision']
for pkg in packages:
    try:
        v = pkg_resources.get_distribution(pkg).version
        print(f"   ✓ {pkg}: {v}")
    except:
        print(f"   ✗ {pkg}: Not found")

# System tools
print("\n System Tools:")
print("\n   Tesseract OCR:")
!tesseract --version | head -1

print("\n Poppler Utils:")
!pdftoppm -v 2>&1 | head -1

# Standard library
print("\n Standard Library (Built-in):")
print("   re (regex)")
print("   json")

print("\n" + "="*70)
